# 04. Tools & Structured Outputs

The model does NOT call backend services directly. Tool calling is a mechanism where the model **proposes** a structured action, and your application **validates, authorizes, and executes** it.

In this lab, we build a **defense-in-depth execution boundary** for a customer support agent assisting users with order inquiries and refunds. We explicitly test that no model proposal can bypass schema constraints, tenant isolation, business rules, or authorization gates.

In [ ]:
import os
import sys

course_dir = os.path.join(
    os.getcwd(),
    "curriculum/beginner/04-tools-and-structured-outputs",
)
if course_dir not in sys.path:
    sys.path.insert(0, course_dir)

from policy import (
    ErrorResult,
    ExecutionContext,
    OrderResult,
    RefundResult,
    crashing_tool_impl,
    dispatch_tool,
    get_order_impl,
    issue_refund_impl
)
import json
from pydantic import BaseModel, Field
from enum import Enum
from typing import Optional, Literal


## Part 1 & 2 — Strict Argument Schemas
We define tool arguments using Pydantic models with `extra='forbid'`. This strictly prevents the model from injecting unauthorized metadata (such as overriding `tenant_id` or `actor_id`).

In [ ]:
class RefundReason(str, Enum):
    DAMAGED = 'damaged'
    LOST = 'lost'
    CUSTOMER_REQUEST = 'customer_request'
# The strict tool schemas have been extracted to policy.py for rigorous unit testing.
import sys, os; sys.path.insert(0, os.path.join(os.getcwd(), 'curriculum/beginner/04-tools-and-structured-outputs')); from policy import GetOrderArgs, IssueRefundArgs
print('IssueRefundArgs JSON Schema:')
print(json.dumps(IssueRefundArgs.model_json_schema(), indent=2))


## Part 3 — Typed Result Models
Tools return strongly typed Pydantic result objects rather than unvalidated dictionaries or raw strings.

> Note: The executable code is imported from `curriculum/beginner/04-tools-and-structured-outputs/policy.py` for parity with regression tests.

```python
class OrderResult(BaseModel):
    order_id: str
    tenant_id: str
    customer_id: str
    total_cents: int
    status: str
```


In [ ]:
print('Result models initialized.')


## Part 4 — Actor Identity & Tenant Scope
The model never provides actor identity or permissions. The application runtime injects a trusted `ExecutionContext` representing the active support agent and organization scope.

- `actor_id`: The support agent invoking the tool (`support-agent-007`).
- `tenant_id`: The organization scope (`northstar`).
- `customer_id`: The end customer who placed the order (`CUST-101`). Orders belong to customers, not the support agent.

> Note: The executable code is imported from `curriculum/beginner/04-tools-and-structured-outputs/policy.py` for parity with regression tests.

```python
class ExecutionContext(BaseModel):
    tenant_id: str
    roles: List[str]
```


In [ ]:

# Context instances for testing authorization boundaries
ctx_support_agent = ExecutionContext(
    actor_id='support-agent-007',
    tenant_id='northstar',
    roles={'order:read', 'refund:issue'}
)
ctx_read_only_agent = ExecutionContext(
    actor_id='trainee-agent-001',
    tenant_id='northstar',
    roles={'order:read'}  # Lacks refund:issue
)
ctx_other_tenant_agent = ExecutionContext(
    actor_id='external-agent-999',
    tenant_id='acme_corp',
    roles={'order:read', 'refund:issue'}
)
print('Execution contexts initialized.')

## Part 5 — Tool Registry with Explicit Permissions & Idempotency
Tools are registered with explicit effect types (`read` vs `write`), required permissions, and argument schemas.

> Note: The executable code is imported from `curriculum/beginner/04-tools-and-structured-outputs/policy.py` for parity with regression tests.

```python
# Mock Database
DB_ORDERS: Dict[str, Dict[str, Any]] = {
    'ORD-123': {'tenant_id': 'northstar', 'customer_id': 'CUST-101', 'total_cents': 5000, 'status': 'delivered'},
    'ORD-456': {'tenant_id': 'northstar', 'customer_id': 'CUST-202', 'total_cents': 10000, 'status': 'delivered'},
    'ORD-789': {'tenant_id': 'other_tenant', 'customer_id': 'CUST-999', 'total_cents': 7500, 'status': 'delivered'}
}
DB_PROCESSED_REFUNDS: Set[str] = set()
# Backend Implementation Handlers
def get_order_impl(args: GetOrderArgs) -> OrderResult:
    rec = DB_ORDERS[args.order_id]
    return OrderResult(
        order_id=args.order_id,
        tenant_id=rec['tenant_id'],
        customer_id=rec['customer_id'],
        total_cents=rec['total_cents'],
        status=rec['status']
    )
def issue_refund_impl(args: IssueRefundArgs) -> RefundResult:
    if args.idempotency_key in DB_PROCESSED_REFUNDS:
        return RefundResult(
            status='already_processed',
            transaction_id='tx_existing_prev',
            amount_cents=args.amount_cents,
            order_id=args.order_id
        )
    DB_PROCESSED_REFUNDS.add(args.idempotency_key)
    return RefundResult(
        status='refund_issued',
        transaction_id='tx_new_8899',
        amount_cents=args.amount_cents,
        order_id=args.order_id
    )
def crashing_tool_impl(args: GetOrderArgs) -> OrderResult:
    raise RuntimeError('Database connection suddenly dropped!')

TOOL_REGISTRY = {
    'get_order': {'schema': GetOrderArgs, 'func': get_order_impl, 'effect': 'read', 'permission': 'order:read'},
    'issue_refund': {'schema': IssueRefundArgs, 'func': issue_refund_impl, 'effect': 'write', 'permission': 'refund:issue'},
    'crash_test': {'schema': GetOrderArgs, 'func': crashing_tool_impl, 'effect': 'read', 'permission': 'order:read'}
}
```


In [ ]:
from typing import Dict
# Mock Database
DB_ORDERS: Dict[str, Dict[str, Any]] = {
    'ORD-123': {'tenant_id': 'northstar', 'customer_id': 'CUST-101', 'total_cents': 5000, 'status': 'delivered'},
    'ORD-456': {'tenant_id': 'northstar', 'customer_id': 'CUST-202', 'total_cents': 10000, 'status': 'delivered'},
    'ORD-789': {'tenant_id': 'other_tenant', 'customer_id': 'CUST-999', 'total_cents': 7500, 'status': 'delivered'}
}
DB_PROCESSED_REFUNDS: Set[str] = set()
# Backend Implementation Handlers
# Tool Capability Registry
TOOL_REGISTRY = {
    'get_order': {'schema': GetOrderArgs, 'func': get_order_impl, 'effect': 'read', 'permission': 'order:read'},
    'issue_refund': {'schema': IssueRefundArgs, 'func': issue_refund_impl, 'effect': 'write', 'permission': 'refund:issue'},
    'crash_test': {'schema': GetOrderArgs, 'func': crashing_tool_impl, 'effect': 'read', 'permission': 'order:read'}
}
print('Tool registry initialized.')


## Part 6 — Defense-in-Depth Safe Dispatcher
The dispatcher enforces the execution boundary in 5 strict phases:
1. **Tool Existence:** Rejects unapproved/unknown tools.
2. **Actor Authorization:** Verifies actor roles against tool permissions.
3. **Schema Validation:** Strictly validates JSON against Pydantic model (`extra='forbid'`).
4. **Tenant & Business Validation:** Checks organization boundary (`order.tenant_id == ctx.tenant_id`) and domain rules (`amount <= total`).
5. **Execution & Idempotency:** Dispatches to backend handler.

> Note: The executable code is imported from `curriculum/beginner/04-tools-and-structured-outputs/policy.py` for parity with regression tests.

```python
def dispatch_tool(tool_name: str, raw_args: str, ctx: ExecutionContext, registry: dict = TOOL_REGISTRY, db_orders: dict = DB_ORDERS) -> BaseModel:
    if tool_name not in registry:
        return ErrorResult(error_type='UNKNOWN_TOOL', message=f"Tool '{tool_name}' not found in registry.")
    
    entry = registry[tool_name]
    req_perm = entry['permission']
    if req_perm not in ctx.roles:
        return ErrorResult(error_type='AUTH_ERROR', message=f"Permission denied: Actor lacks '{req_perm}' role.")
        
    try:
        validated_args = entry['schema'].model_validate_json(raw_args)
    except ValidationError as e:
        err_msg = ', '.join([f"{err['loc'][0]}: {err['msg']}" for err in e.errors()])
        return ErrorResult(error_type='SCHEMA_ERROR', message=err_msg)

    order_id = getattr(validated_args, 'order_id', None)
    if order_id:
        if order_id not in db_orders:
            return ErrorResult(error_type='BUSINESS_ERROR', message=f"Order '{order_id}' not found.")
        order = db_orders[order_id]
        if order['tenant_id'] != ctx.tenant_id:
            return ErrorResult(error_type='AUTH_ERROR', message='Cross-tenant access denied: order belongs to another organization.')
        if tool_name == 'issue_refund':
            if validated_args.amount_cents > order['total_cents']:
                return ErrorResult(error_type='BUSINESS_ERROR', message=f"Refund amount ({validated_args.amount_cents}¢) exceeds order total ({order['total_cents']}¢).")
            
    try:
        return entry['func'](validated_args)
    except Exception as e:
        logging.getLogger(__name__).error(f'Internal unexpected error executing {tool_name}: {e}', exc_info=True)
        return ErrorResult(error_type='INTERNAL_TOOL_ERROR', message='The tool failed unexpectedly.')
```


In [ ]:
print('Safe dispatcher initialized.')


## Part 7 — Evaluation Test Suite
We execute test cases covering normal execution, idempotency, cross-tenant isolation, business limit checks, and unauthorized access.

In [ ]:
import pandas as pd

tests = [
    {'desc': '1. Success - Valid Refund', 'ctx': ctx_support_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_101"}'},
    {'desc': '2. Idempotency - Duplicate Refund', 'ctx': ctx_support_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_101"}'},
    {'desc': '3. Cross-Tenant Denial (Other Org Order)', 'ctx': ctx_support_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-789", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_102"}'},
    {'desc': '4. Business Rule - Refund > Total', 'ctx': ctx_support_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 99000, "reason": "damaged", "idempotency_key": "key_103"}'},
    {'desc': '5. Unknown Tool Proposal', 'ctx': ctx_support_agent, 'tool': 'drop_table', 'args': '{}'},
    {'desc': '6. Parameter Injection Attempt', 'ctx': ctx_support_agent, 'tool': 'get_order', 'args': '{"order_id": "ORD-123", "tenant_id": "override_admin"}'},
    {'desc': '7. Authorization Denied (Trainee Role)', 'ctx': ctx_read_only_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_104"}'},
    {'desc': '8. Unexpected Internal Error', 'ctx': ctx_support_agent, 'tool': 'crash_test', 'args': '{"order_id": "ORD-123"}'}
]

results = []
print('=== EXECUTING VALIDATION SUITE ===')
for t in tests:
    res = dispatch_tool(t['tool'], t['args'], t['ctx'])
    results.append({
        'Test': t['desc'],
        'Result Type': type(res).__name__,
        'Status / Error': getattr(res, 'status', getattr(res, 'error_type', 'UNKNOWN')),
        'Output': res.model_dump_json()
    })

df_tests = pd.DataFrame(results)
print(df_tests.to_string(index=False))

# Assertions verifying all security & business invariants
assert results[0]['Result Type'] == 'RefundResult' and 'refund_issued' in results[0]['Output']
assert results[1]['Result Type'] == 'RefundResult' and 'already_processed' in results[1]['Output']
assert results[2]['Result Type'] == 'ErrorResult' and 'AUTH_ERROR' in results[2]['Output']
assert results[3]['Result Type'] == 'ErrorResult' and 'BUSINESS_ERROR' in results[3]['Output']
assert results[4]['Result Type'] == 'ErrorResult' and 'UNKNOWN_TOOL' in results[4]['Output']
assert results[5]['Result Type'] == 'ErrorResult' and 'SCHEMA_ERROR' in results[5]['Output']
assert results[6]['Result Type'] == 'ErrorResult' and 'AUTH_ERROR' in results[6]['Output']
assert results[7]['Result Type'] == 'ErrorResult' and 'INTERNAL_TOOL_ERROR' in results[7]['Output']
assert 'Database connection suddenly dropped!' not in results[7]['Output']
print('\nAll 8 boundary validation tests passed successfully!')

## Part 8 — Bounded Correction Loop
The application loop permits bounded replanning for correctable `SCHEMA_ERROR`s (e.g. omitted required fields). In contrast, `AUTH_ERROR`s are fatal security boundaries and are **never** resent to the model for retries.

In [ ]:
def simulate_correction_loop(tool_name: str, raw_args: str, ctx: ExecutionContext):
    max_retries = 2
    for attempt in range(1, max_retries + 1):
        print(f'Attempt {attempt}...')
        res = dispatch_tool(tool_name, raw_args, ctx)
        if isinstance(res, ErrorResult):
            if res.error_type == 'SCHEMA_ERROR':
                print(f'-> Schema error: {res.message}. Model receives feedback and corrects arguments...')
                # Simulated model correction on subsequent turn
                raw_args = '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_corrected"}'
            elif res.error_type == 'AUTH_ERROR':
                print(f'-> Security Authorization Denied: {res.message}. Hard stop!')
                break
            else:
                print(f'-> Business error: {res.message}. Hard stop.')
                break
        else:
            print('-> Success:', res.model_dump_json())
            break

print('--- Recoverable Schema Error ---')
simulate_correction_loop('issue_refund', '{"order_id": "ORD-123"}', ctx_support_agent)
print('\n--- Unrecoverable Auth Error ---')
simulate_correction_loop('issue_refund', '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_x"}', ctx_read_only_agent)

## Part 9 — Structured Outputs (Separate from Tool Calling)
Emitting a final structured decision object (e.g. `SupportDecision`) is distinct from calling a backend tool to perform an action.

In [ ]:
class SupportDecision(BaseModel):
    category: Literal['refund', 'replacement', 'escalate']
    summary: str
    requires_human_review: bool
    customer_sentiment: Literal['positive', 'neutral', 'negative']

model_final_output = '{"category": "refund", "summary": "Processed $10.00 refund for damaged item.", "requires_human_review": false, "customer_sentiment": "neutral"}'
decision = SupportDecision.model_validate_json(model_final_output)
print('Validated SupportDecision Object:', decision)

## Part 10 & 11 — Optional Live Model Integration (OpenAI Responses API)
*(Optional)* When `OPENAI_API_KEY` is present, we execute live Tool Calling through our safe dispatcher using the current **OpenAI Responses API** (`client.responses.create`), and test Real Structured Outputs using `client.responses.parse`.

In [ ]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print('No OPENAI_API_KEY detected in environment. Skipping live OpenAI API calls.')
else:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    # 1. Real Tool Calling Hooked to Secure Dispatcher
    print(f'\n--- Live Tool Calling with Responses API ({OPENAI_MODEL}) ---')
    openai_tools = [{
        'type': 'function',
        'name': 'get_order',
        'description': 'Look up order details for a specific customer order ID',
        'parameters': GetOrderArgs.model_json_schema()
    }]
    conversation_input = [{'role': 'user', 'content': 'Can you look up customer order ORD-123?'}]

    response = client.responses.create(model=OPENAI_MODEL, input=conversation_input, tools=openai_tools)
    fn_calls = [item for item in response.output if getattr(item, 'type', None) == 'function_call' or hasattr(item, 'call_id')]

    if fn_calls:
        tc = fn_calls[0]
        try:
            args = json.loads(tc.arguments) if isinstance(tc.arguments, str) else tc.arguments
        except Exception:
            args = {}
        print(f'Model proposed tool: {tc.name} with args: {args}')

        # Dispatch strictly through our defense-in-depth dispatcher
        dispatch_result = dispatch_tool(tc.name, json.dumps(args) if isinstance(args, dict) else args, ctx_support_agent)
        print('Dispatcher Output:', dispatch_result.model_dump_json())

        call_id = getattr(tc, 'call_id', 'call_01')
        conversation_input.append({'type': 'function_call', 'call_id': call_id, 'name': tc.name, 'arguments': json.dumps(args)})
        conversation_input.append({
            'type': 'function_call_output',
            'call_id': call_id,
            'output': dispatch_result.model_dump_json()
        })

        final_res = client.responses.create(model=OPENAI_MODEL, input=conversation_input)
        print('\nFinal Model Response:', final_res.output_text)

    # 2. Live Structured Output Parsing
    print(f'\n--- Live Structured Outputs ({OPENAI_MODEL}) ---')
    try:
        parsed_response = client.responses.parse(
            model=OPENAI_MODEL,
            input='My order ORD-123 arrived completely shattered. I would like a refund please.',
            text_format=SupportDecision
        )
        parsed_decision = parsed_response.output_parsed
        print('Parsed SupportDecision Object (Responses API):', parsed_decision)
    except Exception as e:
        parsed_response = client.beta.chat.completions.parse(
            model=OPENAI_MODEL,
            messages=[{'role': 'user', 'content': 'My order ORD-123 arrived completely shattered. I would like a refund please.'}],
            response_format=SupportDecision
        )
        parsed_decision = parsed_response.choices[0].message.parsed
        print('Parsed SupportDecision Object (Chat Completions fallback):', parsed_decision)